# 01 — Document Ingestion
**Phase:** Foundation — Day 1

**What this notebook does:**
- Loads a PDF using PyMuPDF
- Splits text into overlapping chunks
- Converts chunks to embeddings using sentence-transformers
- Stores everything in ChromaDB (persisted to disk)

**Run this first before any other notebook.**

In [ ]:
import fitz                          # PyMuPDF
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os

load_dotenv()
print('All imports successful')

## Step 1 — Load the PDF
PyMuPDF opens the PDF page by page and extracts all text into one string.

In [ ]:
def load_pdf(path):
    doc = fitz.open(path)
    full_text = ""
    for page_num, page in enumerate(doc):
        text = page.get_text()
        full_text += f"\n[Page {page_num + 1}]\n{text}"
    doc.close()
    return full_text

# ---- CHANGE THIS TO YOUR PDF PATH ----
PDF_PATH = "../docs/sample.pdf"

raw_text = load_pdf(PDF_PATH)

print(f"Total characters loaded : {len(raw_text):,}")
print(f"Estimated word count    : {len(raw_text.split()):,}")
print(f"\nFirst 500 characters:\n{'-'*40}")
print(raw_text[:500])

## Step 2 — Chunk the text
We split into 600-character chunks with 100-character overlap.

**Why overlap?** If a sentence is cut at a chunk boundary, it still appears
complete in the next chunk — preserving context.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "],
    length_function=len
)

chunks = splitter.split_text(raw_text)

print(f"Total chunks created: {len(chunks)}")
print(f"Avg chunk length    : {sum(len(c) for c in chunks) // len(chunks)} chars")
print(f"\n--- Chunk 0 ---\n{chunks[0]}")
print(f"\n--- Chunk 1 (notice overlap with chunk 0) ---\n{chunks[1]}")

## Step 3 — Create embeddings
Each chunk becomes a vector of 384 numbers that captures its meaning.
Chunks with similar meaning will have similar vectors.

In [ ]:
# Downloads ~80MB on first run, cached after that
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding chunks... (this takes 1-2 minutes on first run)")
embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"  → {embeddings.shape[0]} chunks, each with {embeddings.shape[1]} dimensions")
print(f"\nSample — first 8 numbers of chunk 0:")
print(embeddings[0][:8].round(4))

## Step 4 — Store in ChromaDB
`PersistentClient` saves to disk so you never re-embed again.

In [ ]:
client = chromadb.PersistentClient(path="../chroma_db")

# Clear old collection if re-running this notebook
try:
    client.delete_collection("my_docs")
    print("Cleared old collection")
except:
    pass

collection = client.create_collection(
    name="my_docs",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for text
)

# Add in batches to avoid memory issues with large PDFs
BATCH_SIZE = 50
for i in range(0, len(chunks), BATCH_SIZE):
    batch_chunks = chunks[i:i+BATCH_SIZE]
    batch_embeds = embeddings[i:i+BATCH_SIZE]
    batch_ids    = [f"chunk_{j}" for j in range(i, i+len(batch_chunks))]
    collection.add(
        documents=batch_chunks,
        embeddings=batch_embeds.tolist(),
        ids=batch_ids
    )

print(f"Stored {collection.count()} chunks in ChromaDB")
print(f"Database saved to: ../chroma_db/")

## Step 5 — Test retrieval
Verify the right chunks come back for a query. Lower distance = more relevant.

In [ ]:
def test_retrieval(query, top_k=3):
    query_vec = embed_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k
    )
    print(f"Query: {query}")
    print("-" * 50)
    for i, (doc, cid, dist) in enumerate(
        zip(results['documents'][0], results['ids'][0], results['distances'][0])
    ):
        print(f"\nRank {i+1} | {cid} | distance: {dist:.4f}")
        print(doc[:300])

# Change this query to something from YOUR document
test_retrieval("What is the main topic of this document?")

## Key observations
- Distance < 0.4 = strong match
- Distance > 1.0 = weak match — try rephrasing your query
- ChromaDB saved to `../chroma_db/` — do not delete this folder

**Next:** `02_vector_store_retrieval.ipynb` — explore the vector store and understand what's stored.